# US-05 — PDF extraction spike (visual demo)

This notebook is the *eyeball* companion to `tests/ingestion/test_extraction_setup.py`.
It runs the hybrid extraction toolchain on the smallest construction guide
(`PCBUs-Working-Together-GPG.pdf`) so we can see the output before trusting it:

- **PyMuPDF (fitz)** for prose,
- **pdfplumber** for tables.

Run it with the project virtualenv kernel (`.venv`). Paths come from
`src.config.settings`, never hard-coded.

In [1]:
# Make the repo root importable so `src.config` resolves from notebooks/.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import fitz  # PyMuPDF
import pdfplumber
from src.config import settings

print("PyMuPDF", fitz.pymupdf_version)
print("pdfplumber", pdfplumber.__version__)
print("sample PDF exists:", settings.SAMPLE_PDF.exists())

PyMuPDF 1.28.0
pdfplumber 0.11.10
sample PDF exists: True


## 1. PyMuPDF — prose on a single page (page 7)

In [2]:
with fitz.open(str(settings.SAMPLE_PDF)) as doc:
    prose = doc[7].get_text()

print(f"{len(prose)} characters\n")
print(prose[:800])

2785 characters

6
1.0 Understanding your duties under HSWA
What you need to know before reading this guide
If you share duties with other PCBUs, what must you do?
You must consult, cooperate with and coordinate activities with all other  
PCBUs you share duties with, so far as is reasonably practicable.
Where duties are shared, all PCBUs have a responsibility to meet those duties,  
to the extent that they have the ability to inﬂuence or control the matter.
If you’re self-employed
If a self-employed person is working for another PCBU (for example, a self-
employed welder who is contracted by a labour hire company), they both  
share duties as a PCBU. If the self-employed person decides how their own  
work is done and creates and controls risks, they are considered to have the 
ability to inﬂuence or cont


## 2. PyMuPDF — text from every page (not just the first)

In [3]:
with fitz.open(str(settings.SAMPLE_PDF)) as doc:
    pages = [doc[i].get_text() for i in range(doc.page_count)]

non_empty = [i for i, t in enumerate(pages) if t.strip()]
print(f"total pages: {len(pages)}")
print(f"pages with text: {len(non_empty)}")
print(f"char count per page: {[len(t) for t in pages]}")

total pages: 36
pages with text: 36
char count per page: [107, 577, 300, 511, 537, 235, 2103, 2785, 3238, 1260, 3479, 2864, 3106, 2482, 2719, 2690, 2080, 249, 2363, 3061, 2619, 3281, 2755, 2691, 820, 204, 1311, 1549, 1288, 1412, 1685, 1635, 3634, 1902, 1018, 122]


## 4. Full document extraction (the US-05 script)

Runs `extract_all_text` from `src/ingestion/extract.py` — the actual extraction
tool — on the whole document, so we can eyeball that every page produces text.
No cleaning or table reconstruction here; that's US-06 / US-07.

In [4]:
from src.ingestion.extract import extract_all_text

pages = extract_all_text(settings.SAMPLE_PDF)
total_chars = sum(len(text) for text in pages)
print(f"{len(pages)} pages, {total_chars} characters total\n")

for i, text in enumerate(pages):
    print("=" * 70)
    print(f"PAGE {i}  ({len(text.strip())} chars)")
    print("=" * 70)
    print(text.strip() or "[no extractable text — possible image-only page]")
    print()

36 pages, 64672 characters total

PAGE 0  (106 chars)
G O O D P R A C T I C E G U I D E L I N E S
PCBUs  
working  
together
ADVICE WHEN  
CONTRACTING
June 2019

PAGE 1  (576 chars)
ACKNOWLEDGEMENTS
WorkSafe would like to acknowledge and thank the stakeholders 
who have contributed to the development of this guidance.
These guidelines are for Persons Conducting 
a Business or Undertaking (PCBUs) who are 
sharing a workplace with other businesses, 
or are working as part of a contracting chain.
They provide advice on how you can meet 
your duties under the Health and Safety at 
Work Act 2015 (HSWA), illustrate di(erent 
contractual relationships between parties, 
and provide examples of ways you can build 
health and safety into contract management.

PAGE 2  (299 chars)
KEY POINTS
 
–
You must consult, cooperate and coordinate with 
other PCBUs when working in a shared workplace,  
or as part of a contracting chain
 
–
You can’t contract out of health and safety duties
 
–
You should al

## 5. LangChain PyMuPDFLoader — Documents with metadata (comparison spike)

Runs `extract_documents` from `src/ingestion/extract_langchain.py`, which uses
LangChain's **PyMuPDFLoader**. Instead of a bare `list[str]`, it returns one
`Document` per page carrying both the text (`page_content`) and a `metadata`
dict (source, `page`, `total_pages`, PDF provenance) glued together.

Same `fitz` engine underneath, so the **prose is essentially identical** to
section 4 — the difference is the packaged metadata. The final block compares
page counts and total characters against the current `extract_all_text`.

Prose only; tables are deferred to US-07.

In [5]:
from src.ingestion.extract_langchain import extract_documents
from src.ingestion.extract import extract_all_text

docs = extract_documents(settings.SAMPLE_PDF)
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"{len(docs)} Documents, {total_chars} characters total\n")

# Eyeball the packaged metadata + prose per page.
for i, doc in enumerate(docs):
    text = doc.page_content
    print("=" * 70)
    print(f"PAGE {i}  ({len(text.strip())} chars)")
    print(f"metadata: {doc.metadata}")
    print("=" * 70)
    print(text.strip() or "[no extractable text — possible image-only page]")
    print()

# Side-by-side vs the current list[str] extractor (section 4).
string_pages = extract_all_text(settings.SAMPLE_PDF)
string_chars = sum(len(text) for text in string_pages)
print("#" * 70)
print("# PyMuPDFLoader (Documents)  vs  extract_all_text (list[str])")
print("#" * 70)
print(f"{'':22}{'PyMuPDFLoader':>16}{'extract_all_text':>20}")
print(f"{'pages':22}{len(docs):>16}{len(string_pages):>20}")
print(f"{'total characters':22}{total_chars:>16}{string_chars:>20}")

36 Documents, 64636 characters total

PAGE 0  (106 chars)
metadata: {'producer': 'macOS Version 26.5 (Build 25F71) Quartz PDFContext', 'creator': 'Adobe InDesign CC 14.0 (Windows)', 'creationdate': "D:20260722034326Z00'00'", 'source': '/Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI/data/raw/building_and_construction/pcbus_working_together/PCBUs-Working-Together-GPG.pdf', 'file_path': '/Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI/data/raw/building_and_construction/pcbus_working_together/PCBUs-Working-Together-GPG.pdf', 'total_pages': 36, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "D:20260722034326Z00'00'", 'trapped': '', 'modDate': "D:20260722034326Z00'00'", 'creationDate': "D:20260722034326Z00'00'", 'page': 0}
G O O D P R A C T I C E G U I D E L I N E S
PCBUs  
working  
together
ADVICE WHEN  
CONTRACTING
June 2019

PAGE 1  (576 chars)
metadata: {'producer': 'macOS Version 26.5 (Build 25F71) Quartz PDFContext', 'creator

## 6. Unstructured.io cleaning spike (US-06 TEST — one document)

Tests whether Unstructured's hosted Transform API returns clean, chunk-ready text
in one step. It partitions the PDF into typed **elements** (`Title`, `NarrativeText`,
`Header`, `Footer`, `PageNumber`, `Table`, …), so cleaning becomes *drop the noise
element types* rather than regex.

**Setup before running:**
- `unstructured-client` is installed in `.venv` (not pinned — testing only).
- Put your key in a gitignored `.env` at the repo root: `UNSTRUCTURED_API_KEY=...`

**The gate — eyeball three things:**
1. Type counts show `Header`/`Footer`/`PageNumber` were detected (so they get dropped).
2. The cleaned prose has no running headers/footers/page numbers.
3. At least one `Table` comes back with usable `text_as_html`.

Note: hi_res runs layout models server-side, so this is **slower than local `fitz`**.

In [ ]:
import os
from dotenv import load_dotenv
from unstructured_client import UnstructuredClient
from unstructured_client.models import operations, shared

load_dotenv()

client = UnstructuredClient(
    api_key_auth=os.getenv("UNSTRUCTURED_API_KEY"),
    server_url=os.getenv("UNSTRUCTURED_API_URL"),
)

# Point this at your test PDF
filepath = "data/raw/building_and_construction/pcbus_working_together/PCBUs-Working-Together-GPG.pdf"

with open(settings.SAMPLE_PDF, "rb") as f:
    result = client.general.partition(
        request=operations.PartitionRequest(
            partition_parameters=shared.PartitionParameters(
                files=shared.Files(
                    content=f.read(),
                    file_name=os.path.basename(str(settings.SAMPLE_PDF)),
                ),
                strategy=shared.Strategy.HI_RES,
            ),
        ),
    )

# Print first 20 elements to see what comes back
for el in result.elements[:20]:
    print(f"Type: {el['type']:20s} | Text: {el['text'][:100]}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/building_and_construction/pcbus_working_together/PCBUs-Working-Together-GPG.pdf'